In [ ]:
!git clone https://github.com/No-Country-simulation/g9-br-team-34-techmind.git

Cloning into 'g9-br-team-34-techmind'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 62 (delta 6), reused 52 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 208.54 KiB | 1.56 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [ ]:
%cd g9-br-team-34-techmind/

/content/g9-br-team-34-techmind


In [ ]:
!git config --global user.name "Vanzhel01"
!git config --global user.email "huanucoluigui@gmail.com"

In [ ]:
!git checkout Luigui_branch
#Si quieres usar otra branch cambia el nombre en caso de que quieras crear una nueva y usar esa usa:
#!git checkout -b su_propia_branch

Branch 'Luigui_branch' set up to track remote branch 'Luigui_branch' from 'origin'.
Switched to a new branch 'Luigui_branch'


In [ ]:
!git branch

* Luigui_branch
  main


In [2]:
import time
import gzip
import io
import re
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from requests import exceptions


##Configuración

In [3]:
# Regístrate en https://stackapps.com/apps/oauth/register y coloca tu API key aquí para subir la cuota de requests por dia.
STACK_API_KEY = None  # ej: "AbCdEf123..."

SITE = "es.stackoverflow"
PAGESIZE = 100          # máximo
MAX_PAGES_POR_TAG = 3   # ~300 preguntas por tag

# Mapeo: categoría objetivo -> tags reales que existen en stackoverflow en spañol
CATEGORIAS_SO = {
    "Backend":        ["java", "php", "node.js", "api-rest", "spring", "django"],
    "Frontend":       ["javascript", "html", "css", "react.js", "angular", "vue.js"],
    "Base de Datos":  ["mysql", "sql", "sql-server", "postgresql", "mongodb"],
    "Mobile":         ["android", "ios", "flutter", "react-native"],
    "DevOps":         ["docker", "linux", "git", "nginx", "kubernetes"],
    "Machine Learning": ["python", "machine-learning", "pandas", "tensorflow"],
    "Seguridad":      ["seguridad", "seguridad-informática", "encriptación"],
}

# Mapeo: categoría objetivo -> categorías de Wikipedia en español
CATEGORIAS_WIKI = {
    "Backend":        "Categoría:Frameworks de aplicaciones web",
    "Frontend":       "Categoría:Lenguajes de marcado",
    "Base de Datos":  "Categoría:Sistemas_gestores_de_bases_de_datos",
    "DevOps":         "Categoría:Contenedores_(informática)",
    "Machine Learning": "Categoría:Aprendizaje_automático",
    "Seguridad":      "Categoría:Seguridad_informática",
}


##Stack overflow

In [4]:
def limpiar_html(texto_html: str) -> str:
    """Quita etiquetas HTML y bloques de código extensos, deja texto plano legible."""
    soup = BeautifulSoup(texto_html or "", "html.parser")
    # Elimina bloques <pre><code> largos (código fuente) para quedarnos
    # con la explicación en lenguaje natural.
    for pre in soup.find_all("pre"):
        pre.decompose()
    texto = soup.get_text(separator=" ")
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def fetch_preguntas_por_tag(tag: str, max_pages: int = MAX_PAGES_POR_TAG) -> list:
    """Descarga preguntas de es.stackoverflow.com para un tag dado, con paginación y backoff."""
    resultados = []
    page = 1
    while page <= max_pages:
        params = {
            "site": SITE,
            "tagged": tag,
            "page": page,
            "pagesize": PAGESIZE,
            "order": "desc",
            "sort": "votes",
            "filter": "withbody",  # incluye el cuerpo completo de la pregunta
        }
        if STACK_API_KEY:
            params["key"] = STACK_API_KEY

        resp = requests.get(
            "https://api.stackexchange.com/2.3/questions",
            params=params,
            timeout=30,
        )

        # La API de Stack Exchange comprime la respuesta en gzip
        try:
            data = json.loads(gzip.decompress(resp.content))
        except OSError:
            data = resp.json()

        if "error_id" in data:
            print(f"  ! Error API en tag={tag}: {data.get('error_message')}")
            break

        for item in data.get("items", []):
            resultados.append({
                "id": item.get("question_id"),
                "titulo": item.get("title", ""),
                "texto": limpiar_html(item.get("body", "")),
                "tags_originales": ",".join(item.get("tags", [])),
                "fuente": "es.stackoverflow.com",
            })

        # Respeta el backoff sugerido por la API para no ser bloqueado
        backoff = data.get("backoff", 0)
        time.sleep(backoff if backoff else 0.3)

        if not data.get("has_more", False):
            break
        page += 1

    return resultados


def extraer_stackoverflow_es() -> pd.DataFrame:
    filas = []
    for categoria, tags in CATEGORIAS_SO.items():
        print(f"Extrayendo categoría: {categoria}")
        for tag in tags:
            preguntas = fetch_preguntas_por_tag(tag)
            for p in preguntas:
                p["categoria"] = categoria
            filas.extend(preguntas)
            print(f"  tag='{tag}' -> {len(preguntas)} preguntas")

    df = pd.DataFrame(filas)
    df = df.drop_duplicates(subset="id")
    return df

## WIKIPEDIA

In [5]:
def fetch_articulos_wikipedia(categoria_wiki: str, limite: int = 40) -> list:
    """Obtiene el resumen (intro) de artículos pertenecientes a una categoría de Wikipedia en español."""
    session = requests.Session()
    session.headers.update({'User-Agent': 'ColabDatasetExtractor/1.0 (https://colab.research.google.com; user@example.com)'})
    resultados = []

    resp = session.get(
        "https://es.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "list": "categorymembers",
            "cmtitle": categoria_wiki,
            "cmlimit": limite,
            "format": "json",
        },
        timeout=30,
    )
    try:
        data = resp.json()
    except exceptions.JSONDecodeError as e:
        print(f"Error decoding JSON for category members in '{categoria_wiki}': {e}")
        print(f"Status Code: {resp.status_code}")
        print(f"Response Text (first 500 chars): {resp.text[:500]}...")
        return [] # Return empty list to continue processing other categories

    miembros = data.get("query", {}).get("categorymembers", [])

    for m in miembros:
        titulo = m["title"]
        extracto_resp = session.get(
            "https://es.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "prop": "extracts",
                "exintro": True,
                "explaintext": True,
                "titles": titulo,
                "format": "json",
            },
            timeout=30,
        )
        try:
            extracto_data = extracto_resp.json()
        except exceptions.JSONDecodeError as e:
            print(f"Error decoding JSON for article '{titulo}' in category '{categoria_wiki}': {e}")
            print(f"Status Code: {extracto_resp.status_code}")
            print(f"Response Text (first 500 chars): {extracto_resp.text[:500]}...")
            continue # Skip this article and continue with the next

        pages = extracto_data.get("query", {}).get("pages", {})
        for _, pagina in pages.items():
            texto = pagina.get("extract", "").strip()
            if len(texto) > 100:  # descarta artículos casi vacíos/desambiguación
                resultados.append({
                    "id": pagina.get("pageid"),
                    "titulo": titulo,
                    "texto": texto,
                    "tags_originales": categoria_wiki,
                    "fuente": "es.wikipedia.org",
                })
        time.sleep(0.2)

    return resultados


def extraer_wikipedia_es() -> pd.DataFrame:
    filas = []
    for categoria, cat_wiki in CATEGORIAS_WIKI.items():
        print(f"Extrayendo Wikipedia -> categoría: {categoria}")
        articulos = fetch_articulos_wikipedia(cat_wiki)
        for a in articulos:
            a["categoria"] = categoria
        filas.extend(articulos)
        print(f"  {cat_wiki} -> {len(articulos)} artículos")

    df = pd.DataFrame(filas)
    df = df.drop_duplicates(subset="id")
    return df


## Crear dataset balanceado por categorias

In [6]:
def construir_dataset(balancear: bool = True, max_por_categoria: int = 150) -> pd.DataFrame:
    df_so = extraer_stackoverflow_es()
    df_wiki = extraer_wikipedia_es()

    df = pd.concat([df_so, df_wiki], ignore_index=True)
    df = df[df["texto"].str.len() > 30]  # descarta filas con texto muy corto/vacío
    df["fecha_extraccion"] = datetime.now().strftime("%Y-%m-%d")

    if balancear:
        df = (
            df.groupby("categoria", group_keys=False)
            .apply(lambda x: x.sample(min(len(x), max_por_categoria), random_state=42))
            .reset_index(drop=True)
        )

    print("\nDistribución final de clases:")
    print(df["categoria"].value_counts())

    return df

In [7]:
dataset = construir_dataset()

Extrayendo categoría: Backend
  tag='java' -> 300 preguntas
  tag='php' -> 300 preguntas
  tag='node.js' -> 300 preguntas
  tag='api-rest' -> 0 preguntas
  tag='spring' -> 300 preguntas
  tag='django' -> 300 preguntas
Extrayendo categoría: Frontend
  tag='javascript' -> 100 preguntas
  tag='html' -> 300 preguntas
  tag='css' -> 300 preguntas
  tag='react.js' -> 0 preguntas
  tag='angular' -> 300 preguntas
  tag='vue.js' -> 300 preguntas
Extrayendo categoría: Base de Datos
  tag='mysql' -> 300 preguntas
  tag='sql' -> 300 preguntas
  tag='sql-server' -> 300 preguntas
  tag='postgresql' -> 300 preguntas
  tag='mongodb' -> 300 preguntas
Extrayendo categoría: Mobile
  tag='android' -> 300 preguntas
  tag='ios' -> 300 preguntas
  tag='flutter' -> 300 preguntas
  tag='react-native' -> 300 preguntas
Extrayendo categoría: DevOps
  tag='docker' -> 300 preguntas
  tag='linux' -> 300 preguntas
  tag='git' -> 300 preguntas
  tag='nginx' -> 219 preguntas
  tag='kubernetes' -> 38 preguntas
Extrayend

/tmp/ipykernel_992/2420228654.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), max_por_categoria), random_state=42))


In [8]:
dataset

,id,titulo,texto,tags_originales,fuente,categoria,fecha_extraccion
0,4110,&#191;C&#243;mo servir archivos est&#225;ticos...,Estoy desarrollando una aplicación usando Djan...,"python,django",es.stackoverflow.com,Backend,2026-07-28
1,348815,No devuelve la consulta Django Postgresql al l...,apps.urls apps.views El formulario de ingresos...,django,es.stackoverflow.com,Backend,2026-07-28
2,62122,calcular diferencia entre fechas,Estoy tratando de calcular la diferencia entre...,"php,date",es.stackoverflow.com,Backend,2026-07-28
3,403560,Autocompletar input con JavaScript PHP y SQL,"[ACTUALIZADO] Bueno, logre que funcione bien, ...","javascript,php,mysql",es.stackoverflow.com,Backend,2026-07-28
4,285253,"&#191;C&#243;mo admitir letras, n&#250;meros, ...",Necesito que en un campo de formulario sólo se...,"php,regex",es.stackoverflow.com,Backend,2026-07-28
...,...,...,...,...,...,...,...
1045,157536,Verificar la autenticidad de la aplicaci&#243;...,Tengo un problema en lo que a seguridad respec...,"c#,seguridad,servidor",es.stackoverflow.com,Seguridad,2026-07-28
1046,631087,&#191;C&#243;mo corregir &quot;Error 406 - Not...,Tenía este código para obtener el contenido de...,"php,html,curl,seguridad",es.stackoverflow.com,Seguridad,2026-07-28
1047,532523,Desencriptar RSA con llave publica en Swift,Estoy recibiendo un dato encriptado desde back...,"swift,encriptaci&#243;n,rsa",es.stackoverflow.com,Seguridad,2026-07-28
1048,166696,Problema al cifrar y descifrar en cliente y se...,Quiero mandar un mensaje (en este caso simplem...,"java,encriptaci&#243;n,servidor,cifrado",es.stackoverflow.com,Seguridad,2026-07-28


In [9]:
import html

def decode_html_entities(text):
    if isinstance(text, str):
        return html.unescape(text)
    return text

In [10]:
dataset['titulo'] = dataset['titulo'].apply(decode_html_entities)
dataset['texto'] = dataset['texto'].apply(decode_html_entities)
dataset['tags_originales'] = dataset['tags_originales'].apply(decode_html_entities)

In [11]:
dataset

,id,titulo,texto,tags_originales,fuente,categoria,fecha_extraccion
0,4110,¿Cómo servir archivos estáticos desde una apli...,Estoy desarrollando una aplicación usando Djan...,"python,django",es.stackoverflow.com,Backend,2026-07-28
1,348815,No devuelve la consulta Django Postgresql al l...,apps.urls apps.views El formulario de ingresos...,django,es.stackoverflow.com,Backend,2026-07-28
2,62122,calcular diferencia entre fechas,Estoy tratando de calcular la diferencia entre...,"php,date",es.stackoverflow.com,Backend,2026-07-28
3,403560,Autocompletar input con JavaScript PHP y SQL,"[ACTUALIZADO] Bueno, logre que funcione bien, ...","javascript,php,mysql",es.stackoverflow.com,Backend,2026-07-28
4,285253,"¿Cómo admitir letras, números, puntos y guione...",Necesito que en un campo de formulario sólo se...,"php,regex",es.stackoverflow.com,Backend,2026-07-28
...,...,...,...,...,...,...,...
1045,157536,Verificar la autenticidad de la aplicación de ...,Tengo un problema en lo que a seguridad respec...,"c#,seguridad,servidor",es.stackoverflow.com,Seguridad,2026-07-28
1046,631087,"¿Cómo corregir ""Error 406 - Not Acceptable"" al...",Tenía este código para obtener el contenido de...,"php,html,curl,seguridad",es.stackoverflow.com,Seguridad,2026-07-28
1047,532523,Desencriptar RSA con llave publica en Swift,Estoy recibiendo un dato encriptado desde back...,"swift,encriptación,rsa",es.stackoverflow.com,Seguridad,2026-07-28
1048,166696,Problema al cifrar y descifrar en cliente y se...,Quiero mandar un mensaje (en este caso simplem...,"java,encriptación,servidor,cifrado",es.stackoverflow.com,Seguridad,2026-07-28


In [12]:
dataset.to_json("dataset_final.json", index=False)

In [13]:
dataset["categoria"].unique()

array(['Backend', 'Base de Datos', 'DevOps', 'Frontend',
       'Machine Learning', 'Mobile', 'Seguridad'], dtype=object)

In [ ]:
#para hacer los pasos siguientes se debe configurar una llave en git hub y ponerla en la parte de secrets en google colab

#from google.colab import userdata
#token = userdata.get('GITHUB_TOKEN')

In [ ]:
#!git add data-science/
#!git commit -m "agrega estructura base de carpetas para data science"
#!git push https://{token}@github.com/No-Country-simulation/g9-br-team-34-techmind.git Luigui_branch